# Data Prep

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.utils import resample
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier


In [ ]:
df = pd.read_csv('/content/heart_failure_clinical_records.csv')

In [ ]:
df.describe()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,60.288736,0.474400,586.760600,0.439400,37.734600,0.364800,265075.404370,1.369106,136.808200,0.645600,0.311800,130.678800,0.313600
std,11.697243,0.499394,976.733979,0.496364,11.514855,0.481422,97999.758622,1.009750,4.464236,0.478379,0.463275,77.325928,0.464002
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.500000,113.000000,0.000000,0.000000,4.000000,0.000000
25%,50.000000,0.000000,121.000000,0.000000,30.000000,0.000000,215000.000000,0.900000,134.000000,0.000000,0.000000,74.000000,0.000000
50%,60.000000,0.000000,248.000000,0.000000,38.000000,0.000000,263358.030000,1.100000,137.000000,1.000000,0.000000,113.000000,0.000000
75%,68.000000,1.000000,582.000000,1.000000,45.000000,1.000000,310000.000000,1.400000,140.000000,1.000000,1.000000,201.000000,1.000000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.400000,148.000000,1.000000,1.000000,285.000000,1.000000


In [ ]:
df

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,55.0,0,748,0,45,0,263358.03,1.3,137,1,1,88,0
1,65.0,0,56,0,25,0,305000.00,5.0,130,1,0,207,0
2,45.0,0,582,1,38,0,319000.00,0.9,140,0,0,244,0
3,60.0,1,754,1,40,1,328000.00,1.2,126,1,0,90,0
4,95.0,1,582,0,30,0,461000.00,2.0,132,1,0,50,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,45.0,0,582,1,55,0,543000.00,1.0,132,0,0,250,0
4996,60.0,1,582,0,30,1,127000.00,0.9,145,0,0,95,0
4997,95.0,1,112,0,40,1,196000.00,1.0,138,0,0,24,1
4998,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       5000 non-null   float64
 1   anaemia                   5000 non-null   int64  
 2   creatinine_phosphokinase  5000 non-null   int64  
 3   diabetes                  5000 non-null   int64  
 4   ejection_fraction         5000 non-null   int64  
 5   high_blood_pressure       5000 non-null   int64  
 6   platelets                 5000 non-null   float64
 7   serum_creatinine          5000 non-null   float64
 8   serum_sodium              5000 non-null   int64  
 9   sex                       5000 non-null   int64  
 10  smoking                   5000 non-null   int64  
 11  time                      5000 non-null   int64  
 12  DEATH_EVENT               5000 non-null   int64  
dtypes: float64(3), int64(10)
memory usage: 507.9 KB


In [ ]:
df['DEATH_EVENT'].value_counts()

DEATH_EVENT
0    3432
1    1568
Name: count, dtype: int64

In [ ]:
# Separate majority and minority classes
df_majority = df[df.DEATH_EVENT == 0]
df_minority = df[df.DEATH_EVENT == 1]

# Upsample minority class
df_minority_upsampled = resample(df_minority,
                                 replace=True,     # sample with replacement
                                 n_samples=len(df_majority),    # to match majority class
                                 random_state=123) # reproducible results

# Combine majority class with upsampled minority class
df_balanced = pd.concat([df_majority, df_minority_upsampled])

# Display new class counts
df_balanced['DEATH_EVENT'].value_counts()

DEATH_EVENT
0    3432
1    3432
Name: count, dtype: int64

In [ ]:
X = df.drop(columns=['DEATH_EVENT'])
y = df['DEATH_EVENT']

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Fit and transform the features
normalized_features = scaler.fit_transform(X)

# Create a DataFrame with the normalized features
df_norm = pd.DataFrame(normalized_features, columns=X.columns)

# Add the target column back to the normalized DataFrame
df_norm['DEATH_EVENT'] = y.values

# Print the normalized DataFrame
df_norm

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,0.272727,0.0,0.092498,0.0,0.469697,0.0,0.288833,0.089888,0.685714,1.0,1.0,0.298932,0
1,0.454545,0.0,0.004210,0.0,0.166667,0.0,0.339314,0.505618,0.485714,1.0,0.0,0.722420,0
2,0.090909,0.0,0.071319,1.0,0.363636,0.0,0.356286,0.044944,0.771429,0.0,0.0,0.854093,0
3,0.363636,1.0,0.093264,1.0,0.393939,1.0,0.367196,0.078652,0.371429,1.0,0.0,0.306050,0
4,1.000000,1.0,0.071319,0.0,0.242424,0.0,0.528428,0.168539,0.542857,1.0,0.0,0.163701,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0.090909,0.0,0.071319,1.0,0.621212,0.0,0.627834,0.056180,0.542857,0.0,0.0,0.875445,0
4996,0.363636,1.0,0.071319,0.0,0.242424,1.0,0.123530,0.044944,0.914286,0.0,0.0,0.323843,0
4997,1.000000,1.0,0.011355,0.0,0.393939,1.0,0.207177,0.056180,0.714286,0.0,0.0,0.071174,1
4998,0.454545,1.0,0.017479,1.0,0.090909,0.0,0.365984,0.247191,0.085714,0.0,0.0,0.014235,1


In [ ]:
X = df_balanced.drop(columns=['DEATH_EVENT'])
y = df_balanced['DEATH_EVENT']

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Fit and transform the features
normalized_features = scaler.fit_transform(X)

# Create a DataFrame with the normalized features
df_norm_balanced = pd.DataFrame(normalized_features, columns=X.columns)

# Add the target column back to the normalized DataFrame
df_norm_balanced['DEATH_EVENT'] = y.values

# Print the normalized DataFrame
df_norm_balanced

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,0.272727,0.0,0.092498,0.0,0.469697,0.0,0.288833,0.089888,0.685714,1.0,1.0,0.298932,0
1,0.454545,0.0,0.004210,0.0,0.166667,0.0,0.339314,0.505618,0.485714,1.0,0.0,0.722420,0
2,0.090909,0.0,0.071319,1.0,0.363636,0.0,0.356286,0.044944,0.771429,0.0,0.0,0.854093,0
3,0.363636,1.0,0.093264,1.0,0.393939,1.0,0.367196,0.078652,0.371429,1.0,0.0,0.306050,0
4,0.545455,0.0,0.026665,1.0,0.242424,0.0,0.335677,0.078652,0.542857,1.0,0.0,0.733096,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6859,0.545455,1.0,0.015310,0.0,0.696970,0.0,0.395078,0.089888,0.685714,0.0,0.0,0.306050,1
6860,0.636364,1.0,0.028451,1.0,0.015152,0.0,0.209601,0.078652,0.685714,1.0,0.0,0.021352,1
6861,0.181818,1.0,0.294846,1.0,0.318182,0.0,0.060492,0.044944,0.828571,0.0,0.0,0.434164,1
6862,0.545455,0.0,0.012631,1.0,0.469697,1.0,0.313856,0.089888,0.657143,1.0,1.0,0.078292,1


# Splitting the data

In [ ]:
X = df.drop(columns=['DEATH_EVENT'])
y = df['DEATH_EVENT']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_balanced = df_balanced.drop(columns=['DEATH_EVENT'])
y_balanced = df_balanced['DEATH_EVENT']
X_train_balanced, X_test_balanced, y_train_balanced, y_test_balanced = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42)

In [ ]:
X_norm = df_norm.drop(columns=['DEATH_EVENT'])
y_norm = df_norm['DEATH_EVENT']
X_train_norm, X_test_norm, y_train_norm, y_test_norm = train_test_split(X_norm, y_norm, test_size=0.2, random_state=42)

In [ ]:
X_norm_balanced = df_norm.drop(columns=['DEATH_EVENT'])
y_norm_balanced = df_norm['DEATH_EVENT']
X_train_norm_balanced, X_test_norm_balanced, y_train_norm_balanced, y_test_norm_balanced = train_test_split(X_norm_balanced, y_norm_balanced, test_size=0.2, random_state=42)

# applying models

##Applying model (Normal Dataset)

In [ ]:
# Feature selection
selector = SelectKBest(f_classif, k=12)  # Use 'k' to specify the number of features to keep
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Initialize the KNeighborsClassifier
clf = KNeighborsClassifier()

# Perform k-fold cross-validation on the training set
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_accuracy = cross_val_score(clf, X_train_selected, y_train, cv=kfold, scoring='accuracy')
cv_precision = cross_val_score(clf, X_train_selected, y_train, cv=kfold, scoring='precision_weighted')
cv_recall = cross_val_score(clf, X_train_selected, y_train, cv=kfold, scoring='recall_weighted')
cv_f1 = cross_val_score(clf, X_train_selected, y_train, cv=kfold, scoring='f1_weighted')

# Print cross-validation scores
print("Original Dataset Cross-Validation Accuracy:", cv_accuracy.mean())
print("Original Dataset Cross-Validation Precision:", cv_precision.mean())
print("Original Dataset Cross-Validation Recall:", cv_recall.mean())
print("Original Dataset Cross-Validation F1 Score:", cv_f1.mean())

# Train the classifier on the entire training set
clf.fit(X_train_selected, y_train)

# Make predictions on the testing set
y_pred = clf.predict(X_test_selected)

# Calculate evaluation metrics on the test set
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
conf_matrix = confusion_matrix(y_test, y_pred)

# Print evaluation metrics on the test set
print("Original Dataset Test Set Accuracy:", accuracy)
print("Original Dataset Test Set Precision:", precision)
print("Original Dataset Test Set Recall:", recall)
print("Original Dataset Test Set F1 Score:", f1)
print("Original Dataset Confusion Matrix:")
print(conf_matrix)


Original Dataset Cross-Validation Accuracy: 0.9490000000000001
Original Dataset Cross-Validation Precision: 0.9494434889461271
Original Dataset Cross-Validation Recall: 0.9490000000000001
Original Dataset Cross-Validation F1 Score: 0.9491130836271852
Original Dataset Test Set Accuracy: 0.934
Original Dataset Test Set Precision: 0.9344411830943623
Original Dataset Test Set Recall: 0.934
Original Dataset Test Set F1 Score: 0.9341821441207689
Original Dataset Confusion Matrix:
[[662  36]
 [ 30 272]]


##Applying model with balanced dataset

In [ ]:
# Feature selection
selector_balanced = SelectKBest(f_classif, k=12)  # Use 'k' to specify the number of features to keep
X_train_balanced_selected = selector_balanced.fit_transform(X_train_balanced, y_train_balanced)
X_test_balanced_selected = selector_balanced.transform(X_test_balanced)

# Initialize the KNeighborsClassifier
clf_balanced = KNeighborsClassifier()

# Perform k-fold cross-validation on the training set
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_accuracy_balanced = cross_val_score(clf_balanced, X_train_balanced_selected, y_train_balanced, cv=kfold, scoring='accuracy')
cv_precision_balanced = cross_val_score(clf_balanced, X_train_balanced_selected, y_train_balanced, cv=kfold, scoring='precision_weighted')
cv_recall_balanced = cross_val_score(clf_balanced, X_train_balanced_selected, y_train_balanced, cv=kfold, scoring='recall_weighted')
cv_f1_balanced = cross_val_score(clf_balanced, X_train_balanced_selected, y_train_balanced, cv=kfold, scoring='f1_weighted')

# Print cross-validation scores
print("Balanced Dataset Cross-Validation Accuracy:", cv_accuracy_balanced.mean())
print("Balanced Dataset Cross-Validation Precision:", cv_precision_balanced.mean())
print("Balanced Dataset Cross-Validation Recall:", cv_recall_balanced.mean())
print("Balanced Dataset Cross-Validation F1 Score:", cv_f1_balanced.mean())

# Train the classifier on the entire training set
clf_balanced.fit(X_train_balanced_selected, y_train_balanced)

# Make predictions on the testing set
y_pred_balanced = clf_balanced.predict(X_test_balanced_selected)

# Calculate evaluation metrics on the test set
accuracy_balanced = accuracy_score(y_test_balanced, y_pred_balanced)
precision_balanced = precision_score(y_test_balanced, y_pred_balanced, average='weighted')
recall_balanced = recall_score(y_test_balanced, y_pred_balanced, average='weighted')
f1_balanced = f1_score(y_test_balanced, y_pred_balanced, average='weighted')
conf_matrix_balanced = confusion_matrix(y_test_balanced, y_pred_balanced)

# Print evaluation metrics on the test set
print("Balanced Dataset Test Set Accuracy:", accuracy_balanced)
print("Balanced Dataset Test Set Precision:", precision_balanced)
print("Balanced Dataset Test Set Recall:", recall_balanced)
print("Balanced Dataset Test Set F1 Score:", f1_balanced)
print("Balanced Dataset Confusion Matrix:")
print(conf_matrix_balanced)


Balanced Dataset Cross-Validation Accuracy: 0.9522858089087597
Balanced Dataset Cross-Validation Precision: 0.9524248786932483
Balanced Dataset Cross-Validation Recall: 0.9522858089087597
Balanced Dataset Cross-Validation F1 Score: 0.9522866585075827
Balanced Dataset Test Set Accuracy: 0.9431900946831755
Balanced Dataset Test Set Precision: 0.9436036376985714
Balanced Dataset Test Set Recall: 0.9431900946831755
Balanced Dataset Test Set F1 Score: 0.9431967249888135
Balanced Dataset Confusion Matrix:
[[653  49]
 [ 29 642]]


##Applying with normalization

In [ ]:
# Feature selection
selector_norm = SelectKBest(f_classif, k=12)  # Use 'k' to specify the number of features to keep
X_train_norm_selected = selector_norm.fit_transform(X_train_norm, y_train_norm)
X_test_norm_selected = selector_norm.transform(X_test_norm)

# Initialize the KNeighborsClassifier
clf_norm = KNeighborsClassifier()

# Perform k-fold cross-validation on the training set
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
cv_accuracy_norm = cross_val_score(clf_norm, X_train_norm_selected, y_train_norm, cv=kfold, scoring='accuracy')
cv_precision_norm = cross_val_score(clf_norm, X_train_norm_selected, y_train_norm, cv=kfold, scoring='precision_weighted')
cv_recall_norm = cross_val_score(clf_norm, X_train_norm_selected, y_train_norm, cv=kfold, scoring='recall_weighted')
cv_f1_norm = cross_val_score(clf_norm, X_train_norm_selected, y_train_norm, cv=kfold, scoring='f1_weighted')

# Print cross-validation scores
print("Normalized Dataset Cross-Validation Accuracy:", cv_accuracy_norm.mean())
print("Normalized Dataset Cross-Validation Precision:", cv_precision_norm.mean())
print("Normalized Dataset Cross-Validation Recall:", cv_recall_norm.mean())
print("Normalized Dataset Cross-Validation F1 Score:", cv_f1_norm.mean())

# Train the classifier on the entire training set
clf_norm.fit(X_train_norm_selected, y_train_norm)

# Make predictions on the testing set
y_pred_norm = clf_norm.predict(X_test_norm_selected)

# Calculate evaluation metrics on the test set
accuracy_norm = accuracy_score(y_test_norm, y_pred_norm)
precision_norm = precision_score(y_test_norm, y_pred_norm, average='weighted')
recall_norm = recall_score(y_test_norm, y_pred_norm, average='weighted')
f1_norm = f1_score(y_test_norm, y_pred_norm, average='weighted')
conf_matrix_norm = confusion_matrix(y_test_norm, y_pred_norm)

# Print evaluation metrics on the test set
print("Normalized Dataset Test Set Accuracy:", accuracy_norm)
print("Normalized Dataset Test Set Precision:", precision_norm)
print("Normalized Dataset Test Set Recall:", recall_norm)
print("Normalized Dataset Test Set F1 Score:", f1_norm)
print("Normalized Dataset Confusion Matrix:")
print(conf_matrix_norm)


Normalized Dataset Cross-Validation Accuracy: 0.9672499999999999
Normalized Dataset Cross-Validation Precision: 0.9673266994476396
Normalized Dataset Cross-Validation Recall: 0.9672499999999999
Normalized Dataset Cross-Validation F1 Score: 0.967078476477764
Normalized Dataset Test Set Accuracy: 0.969
Normalized Dataset Test Set Precision: 0.9689091675407465
Normalized Dataset Test Set Recall: 0.969
Normalized Dataset Test Set F1 Score: 0.9689259353264245
Normalized Dataset Confusion Matrix:
[[685  13]
 [ 18 284]]


## normalized and balanced

In [ ]:
import mlflow
from mlflow import sklearn
from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Start MLflow run
with mlflow.start_run():
    # Feature selection
    selector_norm_balanced = SelectKBest(f_classif, k=12)
    X_train_norm_balanced_selected = selector_norm_balanced.fit_transform(X_train_norm_balanced, y_train_norm_balanced)
    X_test_norm_balanced_selected = selector_norm_balanced.transform(X_test_norm_balanced)

    # Initialize the KNeighborsClassifier
    clf_norm_balanced = KNeighborsClassifier()

    # Perform k-fold cross-validation on the training set
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    cv_accuracy_norm_balanced = cross_val_score(clf_norm_balanced, X_train_norm_balanced_selected, y_train_norm_balanced, cv=kfold, scoring='accuracy')
    cv_precision_norm_balanced = cross_val_score(clf_norm_balanced, X_train_norm_balanced_selected, y_train_norm_balanced, cv=kfold, scoring='precision_weighted')
    cv_recall_norm_balanced = cross_val_score(clf_norm_balanced, X_train_norm_balanced_selected, y_train_norm_balanced, cv=kfold, scoring='recall_weighted')
    cv_f1_norm_balanced = cross_val_score(clf_norm_balanced, X_train_norm_balanced_selected, y_train_norm_balanced, cv=kfold, scoring='f1_weighted')

    # Print cross-validation scores
    print("Normalized and Balanced Dataset Cross-Validation Accuracy:", cv_accuracy_norm_balanced.mean())
    print("Normalized and Balanced Dataset Cross-Validation Precision:", cv_precision_norm_balanced.mean())
    print("Normalized and Balanced Dataset Cross-Validation Recall:", cv_recall_norm_balanced.mean())
    print("Normalized and Balanced Dataset Cross-Validation F1 Score:", cv_f1_norm_balanced.mean())

    # Train the classifier on the entire training set
    clf_norm_balanced.fit(X_train_norm_balanced_selected, y_train_norm_balanced)

    # Make predictions on the testing set
    y_pred_norm_balanced = clf_norm_balanced.predict(X_test_norm_balanced_selected)

    # Calculate evaluation metrics on the test set
    accuracy_norm_balanced = accuracy_score(y_test_norm_balanced, y_pred_norm_balanced)
    precision_norm_balanced = precision_score(y_test_norm_balanced, y_pred_norm_balanced, average='weighted')
    recall_norm_balanced = recall_score(y_test_norm_balanced, y_pred_norm_balanced, average='weighted')
    f1_norm_balanced = f1_score(y_test_norm_balanced, y_pred_norm_balanced, average='weighted')
    conf_matrix_norm_balanced = confusion_matrix(y_test_norm_balanced, y_pred_norm_balanced)

    # Print evaluation metrics on the test set
    print("Normalized and Balanced Dataset Test Set Accuracy:", accuracy_norm_balanced)
    print("Normalized and Balanced Dataset Test Set Precision:", precision_norm_balanced)
    print("Normalized and Balanced Dataset Test Set Recall:", recall_norm_balanced)
    print("Normalized and Balanced Dataset Test Set F1 Score:", f1_norm_balanced)
    print("Normalized and Balanced Dataset Confusion Matrix:")
    print(conf_matrix_norm_balanced)

    # Save the model as a .pkl file using MLflow
    mlflow.sklearn.log_model(clf_norm_balanced, "model")


Normalized and Balanced Dataset Cross-Validation Accuracy: 0.9672499999999999
Normalized and Balanced Dataset Cross-Validation Precision: 0.9673266994476396
Normalized and Balanced Dataset Cross-Validation Recall: 0.9672499999999999
Normalized and Balanced Dataset Cross-Validation F1 Score: 0.967078476477764
Normalized and Balanced Dataset Test Set Accuracy: 0.969
Normalized and Balanced Dataset Test Set Precision: 0.9689091675407465
Normalized and Balanced Dataset Test Set Recall: 0.969
Normalized and Balanced Dataset Test Set F1 Score: 0.9689259353264245
Normalized and Balanced Dataset Confusion Matrix:
[[685  13]
 [ 18 284]]


/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
